In [1]:
import numpy as np
import pandas as pd
import json

In [2]:
np.random.seed(33)
# Sumber 1: Transaksi (CSV) - 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)
# Sumber 2: Data produk (JSON Lines) - 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")
# Sumber 3: Data ulasan (CSV) - tidak semua transaksi memiliki ulasan
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)
print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum

spark = SparkSession.builder.appName("Tugas6-ETL").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
# EXTRACT - membaca tiga sumber data dengan format berbeda
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
for nama, df in [("Transaksi", df_trx), ("Produk", df_produk), ("Ulasan", df_ulasan)]:
    print(f"Data {nama}: {df.count()} baris")
    df.printSchema()

Data Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [5]:
# TRANSFORM - menggabungkan ketiga sumber data
df_gabung = df_trx.join(df_produk, on="product_id", how="inner") \
                  .join(df_ulasan, on="order_id", how="left")

# Menambahkan kolom total_pendapatan
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
print("Jumlah baris setelah join:", df_gabung.count())
df_gabung.select("order_id", "nama_produk", "kategori", "unit_terjual",
                 "harga", "total_pendapatan", "rating").show(5)

Jumlah baris setelah join: 5000
+--------+-----------+------------+------------+------+----------------+------+
|order_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+-----------+------------+------------+------+----------------+------+
|     TX0|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+-----------+------------+------------+------+----------------+------+
only showing top 5 rows



In [6]:
# Membuat kolom ada_ulasan SEBELUM nilai null diisi
df_gabung = df_gabung.withColumn("ada_ulasan", col("rating").isNotNull())
# Mengisi rating yang kosong dengan angka 0
df_bersih = df_gabung.na.fill({"rating": 0})
df_bersih.groupBy("ada_ulasan").count().show()
print("Sisa nilai null pada kolom rating:", df_bersih.filter(col("rating").isNull()).count())

+----------+-----+
|ada_ulasan|count|
+----------+-----+
|      true| 3500|
|     false| 1500|
+----------+-----+

Sisa nilai null pada kolom rating: 0


In [8]:
# LOAD - menyimpan hasil akhir ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

df_bersih.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
)
print("Pipeline ETL selesai - hasil tersimpan di HDFS.")
# Verifikasi struktur folder di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl
# Membaca kembali untuk memastikan data tersimpan utuh
df_hasil = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")
print("Jumlah baris di HDFS:", df_hasil.count())

[Stage 52:>                                                         (0 + 1) / 1]

Pipeline ETL selesai - hasil tersimpan di HDFS.


-rw-r--r--   3 maulzz supergroup          0 2026-09-30 18:55 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - maulzz supergroup          0 2026-09-30 18:55 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 maulzz supergroup      16364 2026-09-30 18:55 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-42ee6b1d-2634-4531-bb96-1bf83a111a59.c000.snappy.parquet
drwxr-xr-x   - maulzz supergroup          0 2026-09-30 18:55 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 maulzz supergroup      11134 2026-09-30 18:55 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-42ee6b1d-2634-4531-bb96-1bf83a111a59.c000.snappy.parquet
drwxr-xr-x   - maulzz supergroup          0 2026-09-30 18:55 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 maulzz supergroup      10393 2026-09-30 18:55 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-42ee6b1d-2634-4531-bb96-1bf83a111a59.c000.snappy.parquet
drwxr-xr-x   - ma

In [9]:
df_hasil.createOrReplaceTempView("hasil_etl")
hasil_e = spark.sql('''
    SELECT kategori,
           COUNT(*) AS jumlah_transaksi,
           SUM(CASE WHEN ada_ulasan THEN 1 ELSE 0 END) AS transaksi_dengan_ulasan,
           ROUND(SUM(CASE WHEN ada_ulasan THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS persen_ulasan
    FROM hasil_etl
    GROUP BY kategori
    ORDER BY persen_ulasan ASC
''')
hasil_e.show()

+------------+----------------+-----------------------+-------------+
|    kategori|jumlah_transaksi|transaksi_dengan_ulasan|persen_ulasan|
+------------+----------------+-----------------------+-------------+
|     Makanan|             536|                    369|        68.84|
|   Kesehatan|             961|                    662|        68.89|
|     Fashion|            1035|                    726|        70.14|
|Rumah Tangga|             815|                    575|        70.55|
|  Elektronik|            1653|                   1168|        70.66|
+------------+----------------+-----------------------+-------------+

